# Structured Outputs and Gate Checks — IT Notice Chain

A structured output gives Python named fields to handle. A gate checks a response before it is passed to the next prompt. In this experiment, Pydantic checks whether an extraction is valid JSON and includes five required string fields. Passing this check does not establish that the facts match the original note.

You will reuse the IT maintenance note and unchanged `get_completion()` helper, request plain JSON, and inspect the gate before writing a notice. Predict the results of missing-field, wrong-type and incorrect-fact examples, then run their local checks. All inputs are included. Offline mode uses labelled supplied responses. Optional local OpenAI mode generates fresh responses; no earlier notebook state is required. Run from the beginning in a fresh kernel.

## 1. Prepare the environment

The companion `requirements.txt` includes Pydantic version 2. Locally, install the requirements in the notebook's kernel environment. The browser kernel can load the available Pydantic package when importing it; if that package is unavailable in your browser, use local JupyterLab. No model request occurs during this setup.


In [ ]:
from pydantic import BaseModel, ValidationError


## 2. Predict before running the chain

The note names Wi-Fi, Tuesday 09:00–10:00, router replacement, wired connections and the IT service desk. Which fields must Stage 1 preserve? Will an incorrect time be rejected merely because it is incorrect? Write your prediction before inspecting the supplied responses.

In [ ]:
maintenance_note = """
Office Wi-Fi will be unavailable on Tuesday
from 09:00 to 10:00 while technicians replace a router.

Wired connections will remain available.
Employees who need help should contact the IT service desk.
"""


## 3. Reuse the shared helper

This is the exact adapter from Section 3.2. Its two-argument interface is unchanged. Offline samples require an exact matching instruction/input pair; unmatched edited prompts do not generate a model response.

In [ ]:
# Shared offline/live request adapter. No API key is stored in this notebook.
# Code origin: get_completion(system_prompt, user_prompt) was introduced in Section 2.3.
# Notebook extension used from Section 3.2: supplied/live mode and interaction records.
# Evaluation notebooks reuse this helper without changing its two-argument interface.
import json
import os
import sys

MODE = "offline"
MODEL = ""
_client = None
request_records = []
OFFLINE_SAMPLES = {}

# Match supplied examples only when both messages are unchanged.
def _request_key(system_prompt, user_prompt):
    return json.dumps([system_prompt.strip(), user_prompt.strip()], ensure_ascii=False)

# Register a supplied teaching response; this does not contact OpenAI.
def register_sample(system_prompt, user_prompt, response):
    OFFLINE_SAMPLES[_request_key(system_prompt, user_prompt)] = response

# Locate one package-root configuration, while retaining standalone notebook use.
def _configuration_file():
    from pathlib import Path
    bases = [Path.cwd()]
    if "__file__" in globals():
        bases.insert(0, Path(__file__).resolve().parent)
    for base in bases:
        for folder in [base, *base.parents]:
            if (folder / "COMPANION_VERSION").is_file():
                return folder / ".env"
    # Standalone notebooks keep their configuration beside the notebook.
    return bases[0] / ".env"

# Load the private configuration and prepare an OpenAI client when live mode is chosen.
def configure_live(model=""):
    global MODE, MODEL, _client
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("The OpenAI SDK needs a standard Python kernel. Keep offline mode in browser Python.")
    try:
        from openai import OpenAI
        from dotenv import load_dotenv
    except ImportError:
        raise RuntimeError("Install openai and python-dotenv in your Python kernel, then rerun setup.") from None
    # Read the key and model from the local file without printing the key.
    load_dotenv(_configuration_file())
    selected_model = model.strip() or os.getenv("OPENAI_MODEL", "").strip()
    if not selected_model or selected_model.startswith("YOUR_"):
        raise ValueError("Set OPENAI_MODEL in your .env file before choosing live mode.")
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if not key or key.startswith("YOUR_"):
        raise ValueError("Copy your own OpenAI key into the .env file before choosing live mode.")
    _client = OpenAI(timeout=45.0, max_retries=0)
    MODEL = selected_model
    MODE = "live"
    print("OpenAI configured from .env. No key is printed.")


def get_completion(system_prompt, user_prompt):
    if MODE == "live":
        if _client is None:
            raise RuntimeError("Run configure_live(MODEL) in a private Python kernel first.")
        try:
            # Send role instructions separately from the user task and data.
            response = _client.responses.create(model=MODEL, instructions=system_prompt, input=user_prompt, max_output_tokens=1000, store=False)
        except Exception:
            raise RuntimeError("API request failed. Check your model, account access and connection; no key has been printed.") from None
        text = response.output_text
        print("[Live OpenAI response]")
    else:
        # Offline mode returns stored text, not a newly generated model answer.
        text = OFFLINE_SAMPLES.get(_request_key(system_prompt, user_prompt))
        if text is None:
            print("[No saved response for these edited instructions]")
            return "No model was called. Use OpenAI mode in a standard Python kernel, or label a response you write yourself."
        print("[Supplied teaching response — no API call]")
    # Keep messages and responses for evaluation; never record the API key.
    request_records.append({"mode": MODE, "model": MODEL if MODE == "live" else "supplied-example", "system_prompt": system_prompt, "user_prompt": user_prompt, "response": text})
    return text

# Release the API client after finishing the experiment.
def clear_live():
    global _client, MODE
    if _client is not None:
        _client.close()
    _client = None
    MODE = "offline"
    print("Live client cleared from this kernel.")


## 4. Define the instructions and structure

The extraction asks for plain JSON without Markdown code fences. The writing instruction prohibits invented sender details. Neither instruction guarantees compliance, so inspect actual outputs. Each Pydantic field below is required and must be a string; empty strings and wrong factual values can still pass.

In [ ]:
extraction_instructions = """
Extract the maintenance facts.

Return only a JSON object with these fields:
affected_service, day_and_time, reason,
alternative, support_contact.

Every value must be a string.
If a detail is missing, use "Not provided".
Return plain JSON only, beginning with { and ending with }.
Do not use Markdown or code fences.
Do not add information or explanatory text.
"""


notice_instructions = """
Write a short IT maintenance notice for employees
using only the supplied facts.

Include a subject line and a clear, friendly message.
Explain the interruption, available alternative
and support contact.

Do not add a sender name, department name or sign-off unless explicitly supplied.
Do not invent missing details.
Do not describe a detail marked "Not provided"
as if it were known.
"""


In [ ]:
from pydantic import BaseModel, ValidationError


class MaintenanceFacts(BaseModel):
    affected_service: str
    day_and_time: str
    reason: str
    alternative: str
    support_contact: str


### Supplied baseline

The following author-written fixture supports offline execution only. It is not a recorded live OpenAI response. Predictions belong above; the outputs become visible when you run the chain.

In [ ]:
# Register a fixed JSON extraction and its supplied notice.
supplied_json = '{\n  "affected_service": "Office Wi-Fi",\n  "day_and_time": "Tuesday, 09:00 to 10:00",\n  "reason": "Router replacement",\n  "alternative": "Wired connections remain available",\n  "support_contact": "IT service desk"\n}'
supplied_notice = "Subject: Office Wi-Fi maintenance on Tuesday\n\nOffice Wi-Fi will be unavailable on Tuesday from 09:00 to 10:00 while technicians replace a router. Wired connections remain available. For help, contact the IT service desk."
register_sample(extraction_instructions, maintenance_note, supplied_json)
register_sample(notice_instructions,
                MaintenanceFacts.model_validate_json(supplied_json).model_dump_json(indent=2),
                supplied_notice)


## 5. Choose offline or local OpenAI mode

Keep `USE_OPENAI=False` in browser Python. For local live mode, install the package-root requirements and save your own `OPENAI_API_KEY` and `OPENAI_MODEL` in `.env` beside `COMPANION_VERSION`. Then set `USE_OPENAI=True` and run from the top. Standalone copies use `.env` beside the notebook. Never include a key in notebook code or output.

One baseline uses at most two model requests: extraction, then notice writing only if the gate passes. Retries are disabled; the helper caps each response at 1,000 output tokens. The later failure demonstrations are local checks and make no additional OpenAI requests.

In [ ]:
USE_OPENAI = False
if USE_OPENAI:
    configure_live()
else:
    print("Offline: supplied responses and actual Python gate checks.")


## 6. Extract and inspect

Run extraction and read the raw response. The response must be a bare JSON object. A Markdown-wrapped object is still rejected; no silent fence stripping or automatic retry is added.

In [ ]:
facts_text = get_completion(extraction_instructions, maintenance_note)
print("RAW EXTRACTION")
print(facts_text)


## 7. Apply the gate

If parsing or field checks fail, the error is displayed and the second request is skipped. Otherwise the checked record is sent as JSON to the notice-writing stage. The code is the same as the book. Compare any final notice with the original note; check that no unsupported signature was added.

In [ ]:
try:
    facts = MaintenanceFacts.model_validate_json(facts_text)

except ValidationError as error:
    print("The extracted facts failed the check.")
    print(error)
    print("Stage 2 was not called.")

else:
    notice = get_completion(
        notice_instructions,
        facts.model_dump_json(indent=2)
    )

    print("EMPLOYEE NOTICE")
    print(notice)


## 8. Experiment with the limits — no model calls

**Predict first:** Remove `support_contact`; change `alternative` to a list; change only the time to 11:00–12:00. Which will pass? Also consider malformed JSON, Markdown code fences, an empty string and an explicit `Not provided` value.

The next cell checks these supplied strings locally. It never calls Stage 2, even for accepted examples. In the real chain, a failed structural check stops Stage 2, while an accepted record can move onward. You still need to check its facts.

In [ ]:
# Change one feature at a time; keep these teaching tests separate from live output.
import json
baseline = json.loads(supplied_json)
missing = dict(baseline)
del missing["support_contact"]
wrong_type = dict(baseline, alternative=["Wired connections"])
wrong_time = dict(baseline, day_and_time="Tuesday, 11:00 to 12:00")
empty_value = dict(baseline, alternative="")
unknown_contact = dict(baseline, support_contact="Not provided")
cases = {
    "Valid record": supplied_json,
    "Missing field": json.dumps(missing),
    "Wrong type": json.dumps(wrong_type),
    "Incorrect time": json.dumps(wrong_time),
    "Empty string": json.dumps(empty_value),
    "Explicitly unavailable": json.dumps(unknown_contact),
    "Malformed JSON": "{broken",
    "Markdown fences": "```json\n" + supplied_json + "\n```"
}
for label, candidate in cases.items():
    try:
        MaintenanceFacts.model_validate_json(candidate)
    except ValidationError:
        print(label + ": rejected — Stage 2 would stop")
    else:
        print(label + ": structure accepted — facts still need checking")


## 9. Explain and save

Why do the wrong time and empty alternative pass? Which rule would you need to detect those problems? Explain why adding a more explicit prompt does not replace checking the returned text. If the live extraction fails, inspect the error instead of repeatedly rerunning the notebook.

Save your prediction, extracted record, final notice or gate failure and your comparisons. Label supplied and live responses clearly. Further output checks can be designed for the project; automatic feedback/retry loops are outside this experiment.

In [ ]:
# Release the optional live client without making another request.
clear_live()
